# Dynamic Reward Demonstration

This notebook demonstrates how to use the DynamicReward class from the GRPO training pipeline. It shows how to initialize the reward function, process examples, and analyze the results.

In [ ]:
import os
import sys
import torch
import asyncio
from typing import List, Dict

# Ensure the project root is in sys.path for imports
project_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from grpo.config import RewardConfig
from grpo.dynamic_reward import DynamicReward
from utils.similarity_checker import SolutionSimilarityChecker

## Initialize the Reward Function

First, we'll set up the reward configuration and initialize the DynamicReward class.

In [ ]:
# Initialize configuration
reward_config = RewardConfig(model_type="dynamic_demo")

# You can customize reward parameters
reward_config.group_diversity_bonus = 2.0
reward_config.base_reward = 3.0

# Print the configuration
print(f"Model type: {reward_config.model_type}")
print(f"Group diversity bonus: {reward_config.group_diversity_bonus}")
print(f"Base reward: {reward_config.base_reward}")

In [ ]:
# Initialize similarity checker
similarity_checker = SolutionSimilarityChecker(reward_config)

# Initialize dynamic reward function
reward_func = DynamicReward(reward_config, similarity_checker)

# Check if stats object is initialized
print(f"Has stats object: {hasattr(reward_func, 'stats')}")

## Example Data

Let's create some example data to test the reward function.

In [ ]:
# Example solution data
solution_examples = [
    {
        "prompt": "Solve the equation: 2x + 3 = 7",
        "completion": "<thinking>\nTo solve the equation 2x + 3 = 7, I need to isolate x.\n\nStep 1: Subtract 3 from both sides.\n2x + 3 - 3 = 7 - 3\n2x = 4\n\nStep 2: Divide both sides by 2.\n2x/2 = 4/2\nx = 2\n\nThe solution is x = 2.\n</thinking>\n\n<answer>\n<step>Subtract 3 from both sides: 2x + 3 - 3 = 7 - 3</step>\n<step>Simplify: 2x = 4</step>\n<step>Divide both sides by 2: 2x/2 = 4/2</step>\n<step>Simplify: x = 2</step>\n\nThe solution is \\boxed{x = 2}.\n</answer>",
        "answer": "x = 2",
        "example_type": "solution"
    },
    {
        "prompt": "Find the derivative of f(x) = x^2 + 3x + 1",
        "completion": "<thinking>\nTo find the derivative of f(x) = x^2 + 3x + 1, I'll use the power rule and the sum rule.\n\nThe power rule states that the derivative of x^n is n*x^(n-1).\nThe derivative of a constant is 0.\n\nSo for f(x) = x^2 + 3x + 1:\n- The derivative of x^2 is 2x^1 = 2x\n- The derivative of 3x is 3\n- The derivative of 1 is 0\n\nAdding these up: f'(x) = 2x + 3\n</thinking>\n\n<answer>\n<step>Use the power rule for x^2: d/dx(x^2) = 2x</step>\n<step>Use the power rule for 3x: d/dx(3x) = 3</step>\n<step>Use the power rule for the constant 1: d/dx(1) = 0</step>\n<step>Add the terms: f'(x) = 2x + 3 + 0 = 2x + 3</step>\n\nThe derivative is \\boxed{f'(x) = 2x + 3}.\n</answer>",
        "answer": "f'(x) = 2x + 3",
        "example_type": "solution"
    }
]

# Example completion data
completion_examples = [
    {
        "prompt": "Continue the solution: Solve the equation: 2x + 3 = 7\n\n<answer>\n<step>Subtract 3 from both sides: 2x + 3 - 3 = 7 - 3</step>\n<step>Simplify: 2x = 4</step>\n",
        "completion": "<step>Divide both sides by 2: 2x/2 = 4/2</step>\n<step>Simplify: x = 2</step>\n\nThe solution is \\boxed{x = 2}.\n</answer>",
        "answer": "x = 2",
        "partial_solution": "<answer>\n<step>Subtract 3 from both sides: 2x + 3 - 3 = 7 - 3</step>\n<step>Simplify: 2x = 4</step>\n",
        "example_type": "completion"
    }
]

# Print example counts
print(f"Solution examples: {len(solution_examples)}")
print(f"Completion examples: {len(completion_examples)}")

## Calculate Rewards

Now let's calculate rewards for our examples.

In [ ]:
# Function to process examples and calculate rewards
def process_examples(examples):
    # Extract completions and other parameters
    completions = [ex["completion"] for ex in examples]
    prompts = [ex["prompt"] for ex in examples]
    answers = [ex["answer"] for ex in examples]
    example_types = [ex["example_type"] for ex in examples]
    
    # Additional parameters for completion examples
    partial_solutions = []
    for ex in examples:
        if "partial_solution" in ex:
            partial_solutions.append(ex["partial_solution"])
        else:
            partial_solutions.append(None)
    
    # Calculate rewards
    rewards = reward_func(completions, 
                          prompts=prompts, 
                          answer=answers, 
                          example_type=example_types,
                          partial_solution=partial_solutions)
    
    return rewards

# Process solution examples
solution_rewards = process_examples(solution_examples)
print("Solution rewards:")
for i, reward in enumerate(solution_rewards):
    print(f"Example {i+1}: {reward}")

# Process completion examples
completion_rewards = process_examples(completion_examples)
print("\nCompletion rewards:")
for i, reward in enumerate(completion_rewards):
    print(f"Example {i+1}: {reward}")

## Analyze Reward Statistics

Let's examine the reward statistics collected during processing.

In [ ]:
# Print reward statistics
if hasattr(reward_func, 'stats'):
    print("Reward Statistics Summary:")
    print(reward_func.stats.get_summary())
    
    print("\nReward Components:")
    for key, value in reward_func.stats.reward_components.items():
        print(f"  {key}: {value}")
    
    if hasattr(reward_func.stats, 'group_stats'):
        print("\nGroup Statistics:")
        for key, value in reward_func.stats.group_stats.items():
            print(f"  {key}: {value}")
    
    if hasattr(reward_func.stats, 'step_stats'):
        print("\nStep Statistics:")
        for key, value in reward_func.stats.step_stats.items():
            print(f"  {key}: {value}")
else:
    print("No statistics available.")

## Visualize Reward Distribution

Let's visualize the distribution of rewards.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Combine all rewards
all_rewards = solution_rewards + completion_rewards

# Create histogram
plt.figure(figsize=(10, 6))
plt.hist(all_rewards, bins=10, alpha=0.7, color='blue')
plt.axvline(np.mean(all_rewards), color='red', linestyle='dashed', linewidth=1, label=f'Mean: {np.mean(all_rewards):.2f}')
plt.title('Distribution of Rewards')
plt.xlabel('Reward Value')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Experiment with Different Reward Parameters

Let's see how changing reward parameters affects the results.

In [ ]:
# Function to test different reward parameters
def test_reward_parameters(diversity_bonus_values, base_reward_values):
    results = []
    
    for diversity_bonus in diversity_bonus_values:
        for base_reward in base_reward_values:
            # Create new config with these parameters
            test_config = RewardConfig(model_type="test")
            test_config.group_diversity_bonus = diversity_bonus
            test_config.base_reward = base_reward
            
            # Initialize new reward function
            test_similarity_checker = SolutionSimilarityChecker(test_config)
            test_reward_func = DynamicReward(test_config, test_similarity_checker)
            
            # Process examples
            test_solution_rewards = test_reward_func(
                [ex["completion"] for ex in solution_examples],
                prompts=[ex["prompt"] for ex in solution_examples],
                answer=[ex["answer"] for ex in solution_examples],
                example_type=[ex["example_type"] for ex in solution_examples]
            )
            
            # Calculate average reward
            avg_reward = sum(test_solution_rewards) / len(test_solution_rewards)
            
            results.append({
                "diversity_bonus": diversity_bonus,
                "base_reward": base_reward,
                "avg_reward": avg_reward
            })
    
    return results

# Test different parameter combinations
diversity_bonus_values = [0.5, 1.0, 2.0, 3.0]
base_reward_values = [1.0, 2.0, 3.0, 4.0]

parameter_results = test_reward_parameters(diversity_bonus_values, base_reward_values)

# Display results in a table
import pandas as pd

results_df = pd.DataFrame(parameter_results)
pivot_table = results_df.pivot(index="diversity_bonus", columns="base_reward", values="avg_reward")
pivot_table

## Visualize Parameter Effects

Let's visualize how different parameters affect the average reward.

In [ ]:
# Create heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(pivot_table, annot=True, cmap="YlGnBu", fmt=".2f")
plt.title("Average Reward by Parameter Combination")
plt.xlabel("Base Reward")
plt.ylabel("Diversity Bonus")
plt.tight_layout()
plt.show()

## Conclusion

This notebook demonstrated how to use the DynamicReward class from the GRPO training pipeline. We've seen how to:

1. Initialize the reward function with different parameters
2. Process different types of examples (solution and completion)
3. Analyze reward statistics
4. Visualize reward distributions
5. Experiment with different reward parameters

The DynamicReward class dynamically selects between different reward types based on the example type, making it versatile for training models on mixed datasets.